# ST-Reason · 00b Feasibility pilot (week 2)

No estimate in the plan is trusted until this notebook measures it. On the 10 pilot videos and 50 pilot questions it:

1. **Records the environment:** GPU, CUDA, RAM, disk and library versions.
2. **Loads every model** in its planned precision, one at a time. For each it records load time, peak GPU memory, peak system RAM, generation speed, and (for Qwen3-VL in fp16) whether outputs overflow to NaN.
3. **Times the CV pipeline:** 5 fps decoding, YOLO26 + ByteTrack, YOLOE-26 with the fixed vocabulary, and Qwen3-VL event extraction on every window.
4. **Runs the first end-to-end pass:** M2 (event list) + strategy A, plus the blind text-only baseline, scored on the 50 questions. It also times a strategy-D-shaped chain (parse → reason → verify).
5. **Applies the go/no-go rules** from the plan, writes `outputs/pilot/feasibility.json`, and updates `config.yaml` (model choices, grid size, pinned model revisions).

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU). Expect 1.5–2.5 hours, mostly model downloads. Every section saves its result as soon as it finishes; after a disconnect, re-run all and finished sections are skipped.

**Accuracy numbers here are not results.** Fifty pilot questions only check that the pipeline works end to end. Nothing from this notebook goes in the paper except the cost measurements.

In [1]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '00b_pilot'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # before torch loads
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'   # local testing only: fake models, no GPU
FORCE = set()                                    # e.g. {'models'} to redo a finished section
PILOT = P['pilot']
FEAS_PATH = PILOT / 'feasibility.json'
FEAS = load_json(FEAS_PATH, default={})

def section_done(name):
    return name in FEAS and name not in FORCE

def save_section(name, value):
    FEAS[name] = value
    FEAS['updated'] = dt.datetime.now().isoformat(timespec='seconds')
    save_json(FEAS, FEAS_PATH)

RUN = Run(params={'step': 'pilot', 'mock': MOCK})
print('sections already done:', [k for k in FEAS if k not in ('updated',)])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 3769f9b696
[08:15:10] INFO  run started: 00b_pilot-20261016-081448-c6ae86
sections already done: ['environment', 'data', 'models', 'cv', 'events']


## 1. Install and record the environment

Each library is installed with a minimum version. The exact versions installed are recorded here and become the pinned versions for every later notebook.

In [2]:
PKGS = ['ultralytics>=8.4.0', 'transformers>=4.57.0', 'accelerate>=1.0', 'bitsandbytes>=0.45',
        'huggingface_hub>=0.30', 'rank_bm25']
if not MOCK:
    pip_install(*PKGS)
import importlib.metadata as md_
versions = {}
for p in ['torch', 'transformers', 'accelerate', 'bitsandbytes', 'ultralytics', 'huggingface_hub', 'opencv-python',
          'numpy', 'pandas']:
    try:
        versions[p] = md_.version(p)
    except Exception:
        versions[p] = None
env = {'sys': sys_info(), 'versions': versions}
if not MOCK:
    import torch
    env['cuda'] = torch.version.cuda
    env['cuda_available'] = torch.cuda.is_available()
    env['bf16_supported'] = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    if not torch.cuda.is_available():
        log('No GPU in this runtime. Switch to a T4 GPU runtime (or Kaggle) and re-run.', 'ERROR')
save_section('environment', env)
for k, v in env['sys'].items():
    print(f'{k:22s} {v}')
print(json.dumps(versions, indent=1))
HAS_TOKEN = hf_login()

[08:15:10] INFO  pip install ultralytics>=8.4.0 transformers>=4.57.0 accelerate>=1.0 bitsandbytes>=0.45 huggingface_hub>=0.30 rank_bm25
runtime                colab
python                 3.13.15
cpus                   2
ram_total_gb           12.67
ram_available_gb       9.36
local_disk_free_gb     20.93
gpu                    Tesla T4
gpu_mem_gb             15.0
driver                 580.82.07
v_torch                2.11.0+cu130
v_transformers         5.18.0
v_ultralytics          8.4.178
v_bitsandbytes         0.50.2
v_pandas               2.2.3
v_numpy                2.1.3
v_cv2                  4.14.0
{
 "torch": "2.11.0+cu130",
 "transformers": "5.18.0",
 "accelerate": "1.15.0",
 "bitsandbytes": "0.50.2",
 "ultralytics": "8.4.178",
 "huggingface_hub": "1.33.0",
 "opencv-python": "5.0.0.93",
 "numpy": "2.1.3",
 "pandas": "2.2.3"
}


## 2. Data checks

Most of these were confirmed in week 1; they are recorded here so `feasibility.json` is complete on its own.

In [3]:
import pandas as pd, numpy as np
card = load_json(P['subsets'] / 'data_card.json', default={})
meta = pd.read_csv(P['ann'] / 'video_meta.csv', dtype={'video_id': str})
SETS = {n: pd.read_csv(P['subsets'] / f'{n}.csv', dtype={'video_id': str})
        for n in ['dev', 'main_grid', 'descriptive', 'pilot']}
dur = meta.set_index('video_id').duration_s
data_check = {
    'nextqa_test_answers_public': True,
    'nextgqa_test_labels': bool(SETS['main_grid'].has_ground.all()),
    'vidor_coverage': card.get('vidor_coverage'),
    'videos_missing': load_json(P['ann'] / 'videos_missing.json', default=[]),
    'star_built': bool((P['data'] / 'videos_star_eval.zip').exists()),
    'video_seconds': {n: round(float(dur.reindex(d.video_id.unique()).sum()), 1) for n, d in SETS.items()},
    'drive_gb': round(sum(f.stat().st_size for f in BASE.rglob('*') if f.is_file()) / 1024 ** 3, 2),
}
save_section('data', data_check)
print(json.dumps(data_check, indent=1, default=str))

{
 "nextqa_test_answers_public": true,
 "nextgqa_test_labels": true,
 "vidor_coverage": {
  "dev": {
   "videos": 92,
   "with_vidor": 92,
   "share": 1.0
  },
  "main_grid": {
   "videos": 189,
   "with_vidor": 0,
   "share": 0.0
  },
  "descriptive": {
   "videos": 129,
   "with_vidor": 0,
   "share": 0.0
  },
  "pilot": {
   "videos": 10,
   "with_vidor": 10,
   "share": 1.0
  },
  "robustness": {
   "videos": 126,
   "with_vidor": 0,
   "share": 0.0
  },
  "intentqa_eval": {
   "videos": 224,
   "with_vidor": 206,
   "share": 0.92
  }
 },
 "videos_missing": [],
 "star_built": true,
 "video_seconds": {
  "dev": 3879.8,
  "main_grid": 7437.5,
  "descriptive": 5095.7,
  "pilot": 397.4
 },
 "drive_gb": 26.04
}


## 3. Shared pilot helpers

Frame decoding, windows, image preparation, robust JSON parsing, and model wrappers. In mock mode (local testing only) the model wrappers return canned outputs.

In [4]:
import cv2
from PIL import Image, ImageDraw

def decode_frames(path, fps_out):
    """Yield (t_seconds, BGR frame) at about fps_out, decoding the video once and streaming."""
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step, i, next_t = 1.0 / fps_out, 0, 0.0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        t = i / src_fps
        if t + 1e-6 >= next_t:
            yield round(t, 3), fr
            next_t += step
        i += 1
    cap.release()

def make_windows(duration, win, stride):
    starts = np.arange(0, max(duration - win, 0) + 1e-6, stride)
    out = [(round(float(s), 2), round(float(min(s + win, duration)), 2)) for s in starts]
    if not out or out[-1][1] < duration - 0.5:
        out.append((round(max(duration - win, 0), 2), round(duration, 2)))
    return out

def to_pil(frame_bgr, max_pixels):
    im = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    w, h = im.size
    scale = min(1.0, (max_pixels / (w * h)) ** 0.5)
    return im.resize((max(28, int(w * scale)), max(28, int(h * scale)))) if scale < 1 else im

PALETTE = [(230, 25, 75), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180), (70, 240, 240)]
def draw_tracks(im, boxes, scale_x, scale_y):
    """Thin coloured outlines with a small corner ID, as specified in Phase 1."""
    d = ImageDraw.Draw(im)
    for b in boxes:
        x1, y1, x2, y2 = [v * s for v, s in zip(b['xyxy'], (scale_x, scale_y, scale_x, scale_y))]
        c = PALETTE[hash(b['id']) % len(PALETTE)]
        d.rectangle([x1, y1, x2, y2], outline=c, width=2)
        d.text((x1 + 2, y1 + 1), b['id'], fill=c)
    return im

def parse_json(text):
    """First JSON object or array in a model reply, or None."""
    if not isinstance(text, str):
        return None
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    for opener, closer in (('{', '}'), ('[', ']')):
        s = text.find(opener)
        while s != -1:
            depth = 0
            for j in range(s, len(text)):
                depth += (text[j] == opener) - (text[j] == closer)
                if depth == 0:
                    try:
                        return json.loads(text[s:j + 1])
                    except json.JSONDecodeError:
                        break
            s = text.find(opener, s + 1)
    return None

def timed(fn, *a, **k):
    t0 = time.time(); out = fn(*a, **k)
    return out, time.time() - t0

In [11]:
# --- model wrappers (real) ---------------------------------------------------------------------
def load_vlm(spec):
    """Qwen3-VL via transformers: returns (model, processor)."""
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    kw = {'device_map': 'auto', 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw['torch_dtype'] = torch.float16                       # the T4 has no bf16
    model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw).eval()
    proc = AutoProcessor.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    return model, proc

def vlm_generate(model, proc, images, prompt, max_new_tokens=200):
    import torch
    msgs = [{'role': 'user', 'content': [*({'type': 'image', 'image': im} for im in images),
                                          {'type': 'text', 'text': prompt}]}]
    inputs = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]

def vlm_has_nan(model, proc, images):
    """One forward pass: True if the logits contain NaN or inf (fp16 overflow check)."""
    import torch
    msgs = [{'role': 'user', 'content': [*({'type': 'image', 'image': im} for im in images),
                                          {'type': 'text', 'text': 'Describe the scene.'}]}]
    inputs = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits
    return bool(torch.isnan(logits).any() or torch.isinf(logits).any())

def load_llm(spec):
    """Text LLM in 4-bit NF4 (or fp16). Returns (model, tokenizer)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    kw = {'device_map': 'auto', 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_use_double_quant=True)
    else:
        kw['torch_dtype'] = torch.float16
    tok = AutoTokenizer.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    try:
        model = AutoModelForCausalLM.from_pretrained(spec['id'], **kw)
    except (ValueError, KeyError):                                   # e.g. Gemma 3 4B is multimodal
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model.eval(), tok

def llm_generate(model, tok, conversations, max_new_tokens=160, temperature=0.0, batch_size=8,
                 max_batch_tokens=6000):
    """Batched chat generation (thinking off for Qwen3), safe on a 15 GB T4.

    Prompts are sorted by length and grouped so that (prompt + output) tokens per batch stay under
    max_batch_tokens. If the GPU still runs out of memory, the batch is halved and retried.
    Outputs are returned in the original order.
    """
    import torch
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
             for m in conversations]
    lens = [len(tok(t)['input_ids']) for t in texts]
    order = sorted(range(len(texts)), key=lambda k: lens[k])
    outs = [None] * len(texts)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    gen.update(dict(do_sample=True, temperature=temperature) if temperature > 0 else dict(do_sample=False))
    i, cap = 0, batch_size
    while i < len(order):
        j = i + 1                                         # always at least one prompt
        while (j < len(order) and j - i < cap and
               (j - i + 1) * (lens[order[j]] + max_new_tokens) <= max_batch_tokens):
            j += 1
        idx = order[i:j]
        try:
            enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True).to(model.device)
            with torch.inference_mode():
                out = model.generate(**enc, **gen)
            for k, d in zip(idx, tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)):
                outs[k] = d
            del enc, out
            i = j
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                raise RuntimeError(f'a single prompt of {lens[idx[0]]} tokens does not fit on the GPU; '
                                   'lower MAX_MEM_WORDS in section 7')
            cap = max(1, len(idx) // 2)
            log(f'GPU memory full at batch {len(idx)}: retrying with batch {cap}', 'WARN')
    return outs

def n_tokens(tok, text):
    return len(tok(text)['input_ids'])

# --- mock versions (local testing only) --------------------------------------------------------
if MOCK:
    class _M:  # stands in for a model
        device = 'cpu'
    def load_vlm(spec):
        time.sleep(0.05); return _M(), None
    def vlm_generate(model, proc, images, prompt, max_new_tokens=200):
        return '{"events": [{"verb": "pick up", "actor_id": "person_1", "object_id": "cup_2", "location": "table"}]}'
    def vlm_has_nan(model, proc, images):
        return False
    def load_llm(spec):
        time.sleep(0.05); return _M(), None
    def llm_generate(model, tok, conversations, max_new_tokens=160, temperature=0.0, batch_size=8,
                     max_batch_tokens=6000):
        rng = random.Random(len(conversations))
        return ['{"answer": "%s", "evidence": ["E1"], "supported": true, "type": "temporal"}' % rng.choice('ABCDE')
                for _ in conversations]
    def n_tokens(tok, text):
        return len(text.split())

## 4. Model load tests

Each model is loaded, measured, used once, and unloaded before the next. A failure is recorded, never fatal. Qwen3-VL-8B (the no-memory baseline) is the largest download; set `TEST_VLM_8B = False` to skip it on a slow day. Gemma 3 needs `HF_TOKEN` and an accepted licence.

In [6]:
TEST_VLM_8B = True
if section_done('models'):
    MODEL_TESTS = FEAS['models']
    print('models: already measured (in FEAS); add "models" to FORCE to redo')
else:
    MODEL_TESTS = {}
    probe_img = to_pil(np.full((480, 640, 3), 128, np.uint8), CFG['cv']['vlm_max_pixels'])
    plan = [('vlm_events', 'vlm'), ('reasoner_main', 'llm'), ('reasoner_small', 'llm'), ('reasoner_alt', 'llm')]
    if TEST_VLM_8B:
        plan.append(('vlm_baseline', 'vlm'))
    for key, kind in plan:
        spec = CFG['models'][key]
        rec = {'id': spec['id'], 'quant': spec.get('quant')}
        if spec.get('gated') and not HAS_TOKEN and not MOCK:
            rec['status'] = 'skipped: needs HF_TOKEN'
            MODEL_TESTS[key] = rec; log(f'{key}: {rec["status"]}', 'WARN'); continue
        try:
            with PeakMemory() as pm:
                (model, proc), t_load = timed(load_vlm if kind == 'vlm' else load_llm, spec)
                rec['load_s'] = round(t_load, 1)
                if kind == 'vlm':
                    if spec.get('quant') == 'fp16':
                        rec['fp16_nan'] = vlm_has_nan(model, proc, [probe_img] * 4)
                    txt, t = timed(vlm_generate, model, proc, [probe_img] * 4,
                                   'List the objects you see as JSON {"objects": [...]}', 64)
                    rec['gen_s_4frames_64tok'] = round(t, 2)
                else:
                    conv = [[{'role': 'user', 'content': 'Count from one to thirty in words, separated by commas.'}]]
                    txt, t = timed(llm_generate, model, proc, conv, 128)
                    ntok = n_tokens(proc, txt[0]) if proc is not None else 128
                    rec['tokens_per_s_single'] = round(ntok / max(t, 1e-6), 1)
                    txt8, t8 = timed(llm_generate, model, proc, conv * 8, 128)
                    rec['tokens_per_s_batch8'] = round(8 * ntok / max(t8, 1e-6), 1)
                rec['sample_output'] = (txt if isinstance(txt, str) else txt[0])[:200]
            rec.update({'status': 'ok', 'peak_vram_gb': pm.peak_vram_gb, 'peak_ram_gb': pm.peak_ram_gb})
        except Exception as e:
            rec.update({'status': f'failed: {e.__class__.__name__}: {str(e)[:300]}'})
        finally:
            for n in ['model', 'proc']:
                globals().pop(n, None)
            free_memory()
        MODEL_TESTS[key] = rec
        log(f"{key}: {rec['status']} | load {rec.get('load_s')}s | VRAM {rec.get('peak_vram_gb')} GB | RAM {rec.get('peak_ram_gb')} GB")
        save_section('models', MODEL_TESTS)                          # save after every model
print(pd.DataFrame(MODEL_TESTS).T.drop(columns=['sample_output'], errors='ignore').to_string())

models: already measured (in FEAS); add "models" to FORCE to redo
                                       id quant  load_s fp16_nan gen_s_4frames_64tok                   status peak_vram_gb peak_ram_gb tokens_per_s_single tokens_per_s_batch8
vlm_events      Qwen/Qwen3-VL-4B-Instruct  fp16   140.0    False                2.67                       ok         8.85        6.53                 NaN                 NaN
reasoner_main               Qwen/Qwen3-8B   nf4   711.5      NaN                 NaN                       ok         5.83        7.31                10.9                54.2
reasoner_small              Qwen/Qwen3-4B   nf4   477.8      NaN                 NaN                       ok         2.68        7.04                11.3                79.3
reasoner_alt         google/gemma-3-4b-it   nf4     NaN      NaN                 NaN  skipped: needs HF_TOKEN          NaN         NaN                 NaN                 NaN
vlm_baseline    Qwen/Qwen3-VL-8B-Instruct   nf4  1055.9    

## 5. CV timing on the pilot videos

Only the 10 pilot videos are extracted from the Drive zip. Each is decoded once at 5 fps. YOLO26 + ByteTrack runs on every decoded frame, and YOLOE-26 with the fixed vocabulary runs on a sample of frames for timing. The tracks are kept so the next step can draw them on the VLM frames.

The fixed vocabulary is built here and saved to `config/openvocab.json`. It comes from a fixed list of everyday objects outside COCO, ranked by how often each appears in NExT-QA **train** questions and options. It is the same for every video and question.

In [7]:
VID_DIR = P['local_videos'] / 'pilot'
VID_DIR.mkdir(parents=True, exist_ok=True)
pilot = SETS['pilot']
pilot_vids = sorted(pilot.video_id.unique())
with zipfile.ZipFile(P['data'] / 'videos_nextqa_eval.zip') as zf:
    names = {Path(n).stem: n for n in zf.namelist()}
    for v in pilot_vids:
        out = VID_DIR / f'{v}.mp4'
        if not out.exists():
            with zf.open(names[v]) as s, open(out, 'wb') as d:
                shutil.copyfileobj(s, d)

# fixed open vocabulary (train questions only, so evaluation questions never shape the memory)
VOCAB_PATH = P['config'] / 'openvocab.json'
COCO = set('person bicycle car motorcycle airplane bus train truck boat traffic light fire hydrant stop sign parking meter bench '
           'bird cat dog horse sheep cow elephant bear zebra giraffe backpack umbrella handbag tie suitcase frisbee skis '
           'snowboard sports ball kite baseball bat glove skateboard surfboard tennis racket bottle wine glass cup fork knife '
           'spoon bowl banana apple sandwich orange broccoli carrot hot dog pizza donut cake chair couch potted plant bed '
           'dining table toilet tv laptop mouse remote keyboard cell phone microwave oven toaster sink refrigerator book '
           'clock vase scissors teddy hair drier toothbrush'.split())
# Fixed candidate list of everyday objects outside COCO-80. The 100 most frequent in NExT-QA *train*
# questions and options become the open vocabulary: deterministic, and never shaped by evaluation questions.
CANDIDATES = sorted(set("""toy toys ball balloon box bag basket blanket pillow towel tissue paper napkin bucket cart stroller
pram swing slide sand rope stick leaf leaves flower flowers plant tree grass rock stone water pool hose bubble bubbles
camera phone tablet guitar piano drum drums violin microphone speaker headphones glasses hat cap helmet shoe shoes sock
socks shirt jacket coat dress scarf glove gloves ring necklace bracelet watch wallet key keys door window curtain mirror
table desk shelf cabinet drawer stairs floor wall carpet mat rug sofa cushion lamp light candle candles fan heater
plate tray pan pot lid jar can mug straw spoon bowl bottle tube cream lotion brush comb towel soap toothpaste sponge
cloth rag broom mop vacuum bin trash food bread rice noodles egg eggs meat fruit candy chocolate cookie biscuit snack
milk juice cereal ice icecream cone popsicle lollipop sweet sweets present gift wrapping ribbon card book books
notebook pen pencil crayon crayons paint marker sticker puzzle block blocks lego doll dolls car cars truck tricycle
scooter skateboard bike horse pony saddle leash collar bone fence gate bench pole flag sign tent boat paddle net
racket bat club stroller seat chair high highchair bottle feeder""".split()) - COCO)
if not VOCAB_PATH.exists():
    from collections import Counter
    train_q = pd.read_csv(P['ann'] / 'questions_nextqa.csv', usecols=['split', 'question', 'a0', 'a1', 'a2', 'a3', 'a4'])
    train_q = train_q[train_q.split == 'train']
    text = ' '.join(train_q.question.tolist() + train_q[['a0', 'a1', 'a2', 'a3', 'a4']].astype(str).values.ravel().tolist())
    freq = Counter(re.findall(r'[a-z]+', text.lower()))
    ranked = sorted((w for w in CANDIDATES if freq[w] >= 5), key=lambda w: (-freq[w], w))
    vocab = ranked[:100]
    save_json({'source': 'fixed everyday-object candidate list ranked by frequency in NExT-QA train questions/options',
               'nouns': vocab, 'counts': {w: freq[w] for w in vocab}, 'coco': sorted(COCO)}, VOCAB_PATH)
OPENVOCAB = load_json(VOCAB_PATH)['nouns']
print(f'open vocabulary: {len(OPENVOCAB)} nouns, e.g. {OPENVOCAB[:12]}')

def load_yolo(names):
    from ultralytics import YOLO
    last = None
    for n in names:
        try:
            return YOLO(n), n
        except Exception as e:
            last = e
    raise last

TRACKS = {}
if section_done('cv') and all((PILOT / 'tracks' / f'{v}.json').exists() for v in pilot_vids):
    CV = FEAS['cv']
    TRACKS = {v: load_json(PILOT / 'tracks' / f'{v}.json') for v in pilot_vids}
    print('cv: already measured')
else:
    CV = {'videos': {}}
    fps_out = CFG['cv']['decode_fps']
    for v in pilot_vids:
        t0 = time.time(); frames = list(decode_frames(VID_DIR / f'{v}.mp4', fps_out)); t_dec = time.time() - t0
        rec = {'frames': len(frames), 'decode_s': round(t_dec, 2)}
        per_frame = []
        if MOCK:
            rng = random.Random(v)
            per_frame = [{'t': t, 'boxes': [{'id': 'person_1', 'cls': 'person', 'xyxy': [10, 10, 100, 200], 'conf': 0.9},
                                             {'id': 'cup_2', 'cls': 'cup', 'xyxy': [120, 80, 150, 120], 'conf': 0.5}]}
                         for t, _ in frames]
            rec['det_track_s'] = 0.01 * len(frames); rec['det_model'] = 'mock'
        else:
            try:
                det, det_name = load_yolo([CFG['cv']['det_model'], 'yolo26m.pt', 'yolo11m.pt'])
                t0 = time.time()
                for t, fr in frames:
                    r = det.track(fr, persist=True, tracker=CFG['cv']['tracker'], conf=CFG['cv']['det_conf'], verbose=False)[0]
                    b = r.boxes
                    ids = b.id.int().tolist() if b.id is not None else [None] * len(b)
                    per_frame.append({'t': t, 'boxes': [
                        {'id': f'{r.names[int(c)].replace(" ", "_")}_{i}' if i is not None else None,
                         'cls': r.names[int(c)], 'xyxy': [round(float(x), 1) for x in xyxy], 'conf': round(float(cf), 3)}
                        for xyxy, c, cf, i in zip(b.xyxy.tolist(), b.cls.tolist(), b.conf.tolist(), ids)]})
                rec['det_track_s'] = round(time.time() - t0, 2); rec['det_model'] = det_name
                del det; free_memory()
            except Exception as e:
                rec['det_error'] = f'{e.__class__.__name__}: {str(e)[:200]}'
        tracked = [bx for f in per_frame for bx in f['boxes'] if bx['id']]
        rec['tracks'] = len({bx['id'] for bx in tracked}); rec['detections'] = sum(len(f['boxes']) for f in per_frame)
        save_json(per_frame, PILOT / 'tracks' / f'{v}.json'); TRACKS[v] = per_frame
        CV['videos'][v] = rec
        log(f"{v}: {rec['frames']} frames, decode {rec['decode_s']}s, det+track {rec.get('det_track_s')}s, {rec['tracks']} tracks")
    # open-vocabulary timing on a sample of frames
    if not MOCK:
        try:
            ov, ov_name = load_yolo([CFG['cv']['openvocab_model'], 'yoloe-26m-seg.pt', 'yoloe-11m-seg.pt'])
            vocab = sorted(set(OPENVOCAB) | COCO)
            try:
                ov.set_classes(vocab, ov.get_text_pe(vocab))
            except TypeError:
                ov.set_classes(vocab)
            sample = [fr for _, fr in list(decode_frames(VID_DIR / f'{pilot_vids[0]}.mp4', 1))[:20]]
            t0 = time.time()
            for fr in sample:
                ov.predict(fr, conf=CFG['cv']['det_conf'], verbose=False)
            CV['openvocab'] = {'model': ov_name, 's_per_frame': round((time.time() - t0) / max(1, len(sample)), 3),
                               'vocab_size': len(vocab)}
            del ov; free_memory()
        except Exception as e:
            CV['openvocab'] = {'error': f'{e.__class__.__name__}: {str(e)[:200]}'}
    vids = CV['videos'].values()
    n_frames = sum(r['frames'] for r in vids)
    CV['s_per_frame_det_track'] = round(sum(r.get('det_track_s', 0) for r in vids) / max(1, n_frames), 4)
    CV['s_per_frame_decode'] = round(sum(r['decode_s'] for r in vids) / max(1, n_frames), 4)
    save_section('cv', CV)
print(json.dumps({k: v for k, v in CV.items() if k != 'videos'}, indent=1))

open vocabulary: 100 nouns, e.g. ['toy', 'camera', 'food', 'water', 'shirt', 'floor', 'microphone', 'watch', 'guitar', 'sofa', 'swing', 'paper']
cv: already measured
{
 "openvocab": {
  "model": "yoloe-26m-seg.pt",
  "s_per_frame": 0.07,
  "vocab_size": 192
 },
 "s_per_frame_det_track": 0.0285,
 "s_per_frame_decode": 0.0066
}


## 6. Event extraction on every window (Qwen3-VL-4B)

Windows are 3 s long with a 2 s stride, and every window is processed. Each window sends 4 frames with the tracks drawn on them. In the pilot, events take their window's times; boundary snapping to tracked state changes comes in notebook 03. One JSON file per video is saved as soon as it is done.

In [8]:
EV_DIR = PILOT / 'events'
EV_DIR.mkdir(exist_ok=True)
EVENT_PROMPT = (
    'These {k} frames come from {ws:.1f}s to {we:.1f}s of a video. Tracked people and objects are outlined, '
    'with their IDs written in the corner (for example person_1, cup_3).\n'
    'List every action that happens in this window. Use only IDs that are shown; use null if an object has no ID.\n'
    'Reply with JSON only: {{"events": [{{"verb": "...", "actor_id": "...", "object_id": "...", "location": "..."}}]}}. '
    'If nothing happens, reply {{"events": []}}.')

def window_images(v, frames, ws, we, k, max_pixels):
    """frames: {t: BGR frame} for the whole video, decoded once per video by the caller."""
    times = sorted(t for t in frames if ws - 1e-6 <= t <= we + 1e-6)
    if not times:
        return []
    pick = [times[int(i)] for i in np.linspace(0, len(times) - 1, k)]
    track_at = {round(f['t'], 3): f['boxes'] for f in TRACKS.get(v, [])}
    out = []
    for t in pick:
        im = to_pil(frames[t], max_pixels)
        h, w = frames[t].shape[:2]
        out.append(draw_tracks(im, [b for b in track_at.get(t, []) if b['id']], im.size[0] / w, im.size[1] / h))
    return out

todo = [v for v in pilot_vids if not (EV_DIR / f'{v}.json').exists()]
EVT = {'windows': 0, 'seconds': 0.0, 'parsed': 0}
if todo:
    vlm_spec = dict(CFG['models']['vlm_events'])
    if MODEL_TESTS.get('vlm_events', {}).get('fp16_nan'):
        vlm_spec['quant'] = 'nf4'
        log('Qwen3-VL-4B overflowed in fp16 during the load test: using 4-bit', 'WARN')
    model, proc = load_vlm(vlm_spec)
    for v in todo:
        d = float(meta.set_index('video_id').duration_s.get(v))
        frames = {round(t, 3): fr for t, fr in decode_frames(VID_DIR / f'{v}.mp4', CFG['cv']['decode_fps'])}
        recs = []
        for ws, we in make_windows(d, CFG['cv']['window_s'], CFG['cv']['window_stride_s']):
            ims = window_images(v, frames, ws, we, CFG['cv']['frames_per_window'], CFG['cv']['vlm_max_pixels'])
            if not ims:
                continue
            txt, t = timed(vlm_generate, model, proc, ims, EVENT_PROMPT.format(k=len(ims), ws=ws, we=we), 200)
            js = parse_json(txt)
            ok = isinstance(js, dict) and isinstance(js.get('events'), list)
            recs.append({'ws': ws, 'we': we, 'events': js['events'] if ok else [], 'parsed': ok, 'raw': None if ok else txt[:300],
                         'seconds': round(t, 2)})
            EVT['windows'] += 1; EVT['seconds'] += t; EVT['parsed'] += ok
        save_json(recs, EV_DIR / f'{v}.json')
        del frames; gc.collect()
        log(f'{v}: {len(recs)} windows, {sum(len(r["events"]) for r in recs)} events')
    del model, proc; free_memory()

all_w = [w for v in pilot_vids for w in load_json(EV_DIR / f'{v}.json', default=[])]
EVENTS_STATS = {'windows': len(all_w), 's_per_window': round(float(np.mean([w['seconds'] for w in all_w])), 2) if all_w else None,
                'json_parse_rate': round(float(np.mean([w['parsed'] for w in all_w])), 3) if all_w else None,
                'events_per_window': round(float(np.mean([len(w['events']) for w in all_w])), 2) if all_w else None}
save_section('events', EVENTS_STATS)
print(EVENTS_STATS)

{'windows': 196, 's_per_window': 10.44, 'json_parse_rate': 0.898, 'events_per_window': 2.48}


## 7. First end-to-end pass: M2 + strategy A, the blind baseline, and a D-shaped chain

- **M2 (event list):** the window events are merged (same verb and IDs, tIoU ≥ 0.5) and numbered E1, E2, … in time order.
- **Strategy A:** memory + question + options → answer letter + cited events, one call per question.
- **Blind baseline:** question + options only, no video, to measure language bias.
- **D-shaped chain** (10 questions): parse → reason → verify, three calls, to time strategy D's real cost including overheads.

Each prediction is appended to a JSONL file as soon as it is made.

In [12]:
LETTERS = 'ABCDE'

def tiou(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0])); union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0

def build_m2(window_recs):
    ev = []
    for w in window_recs:
        for e in w['events']:
            if not isinstance(e, dict) or not e.get('verb'):
                continue
            key = (str(e.get('verb')).lower(), e.get('actor_id'), e.get('object_id'))
            merged = False
            for x in ev:
                if x['key'] == key and tiou((x['t0'], x['t1']), (w['ws'], w['we'])) >= CFG['cv']['merge_min_tiou'] - 1e-9:
                    x['t1'] = max(x['t1'], w['we']); merged = True; break
                if x['key'] == key and abs(x['t1'] - w['ws']) <= CFG['cv']['window_stride_s']:
                    x['t1'] = max(x['t1'], w['we']); merged = True; break
            if not merged:
                ev.append({'key': key, 't0': w['ws'], 't1': w['we'], 'loc': e.get('location')})
    ev.sort(key=lambda x: x['t0'])
    lines = []
    for i, x in enumerate(ev, 1):
        verb, actor, obj = x['key']
        s = f'E{i} [t={x["t0"]:.1f}-{x["t1"]:.1f}s] {actor or "someone"} {verb}' + (f' {obj}' if obj else '')
        lines.append(s + (f' ({x["loc"]})' if x.get('loc') else ''))
    return '\n'.join(lines) if lines else '(no events detected)'

def options_text(r):
    return '\n'.join(f'{LETTERS[i]}. {getattr(r, f"a{i}")}' for i in range(5))

SYS = 'You answer multiple-choice questions about a video using only the memory given. Reply with JSON only.'
def conv_A(memory, r):
    return [{'role': 'system', 'content': SYS},
            {'role': 'user', 'content': f'VIDEO MEMORY (event list):\n{memory}\n\nQUESTION: {r.question}?\nOPTIONS:\n'
                                        f'{options_text(r)}\n\nReply as {{"answer": "A-E", "evidence": ["E1", ...]}}.'}]
def conv_blind(r):
    return [{'role': 'system', 'content': SYS},
            {'role': 'user', 'content': f'QUESTION: {r.question}?\nOPTIONS:\n{options_text(r)}\n\n'
                                        'Reply as {"answer": "A-E"}.'}]
def conv_parse(r):
    return [{'role': 'user', 'content': f'Classify this video question. QUESTION: {r.question}?\nReply as '
             '{"type": "causal|temporal_before|temporal_after|temporal_during|descriptive", "target_object": "...", '
             '"anchor_event": "..."}'}]
def conv_verify(memory, r, letter):
    return [{'role': 'user', 'content': f'MEMORY:\n{memory}\n\nQUESTION: {r.question}?\nPROPOSED ANSWER: '
             f'{getattr(r, "a" + str(LETTERS.index(letter)))}\nIs the proposed answer supported by the memory? '
             'Reply as {"supported": true|false, "evidence": ["E..."]}.'}]

def letter_of(js):
    a = str((js or {}).get('answer', '')).strip().upper()[:1]
    return a if a in LETTERS else None

memories = {v: build_m2(load_json(EV_DIR / f'{v}.json', default=[])) for v in pilot_vids}
E2E_MEM_WORDS = {v: len(m.split(' ')) for v, m in memories.items()}   # real length, before any cut
MAX_MEM_WORDS = 1800        # about 2,500 tokens: keeps every prompt well inside a T4's memory
for v, m in memories.items():
    words = m.split(' ')
    if len(words) > MAX_MEM_WORDS:
        memories[v] = ' '.join(words[:MAX_MEM_WORDS]) + '\n(... memory truncated for the pilot)'
        log(f'{v}: M2 memory had {len(words)} words; truncated to {MAX_MEM_WORDS} for the pilot', 'WARN')
for v, m in memories.items():
    (PILOT / 'memory_M2').mkdir(exist_ok=True)
    (PILOT / 'memory_M2' / f'{v}.txt').write_text(m)

main_key = 'reasoner_main'
if not MODEL_TESTS.get('reasoner_main', {}).get('status', '').startswith('ok'):
    main_key = 'reasoner_small'
    log('Qwen3-8B did not pass the load test: the pilot uses Qwen3-4B', 'WARN')
llm_spec = CFG['models'][main_key]

PRED_A, PRED_B = PILOT / 'pred_M2_A.jsonl', PILOT / 'pred_blind.jsonl'
rows = list(pilot.itertuples(index=False))
need_A = [r for r in rows if r.uid not in done_ids(PRED_A)]
need_B = [r for r in rows if r.uid not in done_ids(PRED_B)]
E2E = FEAS.get('e2e', {})
if need_A or need_B or 'D_chain_s_per_q' not in E2E:
    model, tok = load_llm(llm_spec)
    for name, need, path, mk in [('A', need_A, PRED_A, lambda r: conv_A(memories[r.video_id], r)),
                                 ('blind', need_B, PRED_B, conv_blind)]:
        t0 = time.time()
        for i in range(0, len(need), 8):
            chunk = need[i:i + 8]
            outs = llm_generate(model, tok, [mk(r) for r in chunk], CFG['llm']['max_new_tokens'])
            for r, o in zip(chunk, outs):
                js = parse_json(o); L = letter_of(js)
                append_jsonl(path, {'uid': r.uid, 'pred': L, 'gold': LETTERS[int(r.answer_idx)], 'valid_json': js is not None,
                                    'evidence': (js or {}).get('evidence'), 'raw': None if js else o[:200]})
        if need:
            E2E[f'{name}_s_per_q_batched'] = round((time.time() - t0) / len(need), 2)
    # D-shaped chain on 10 questions, unbatched, to measure the real per-question cost
    t0 = time.time()
    for r in rows[:10]:
        llm_generate(model, tok, [conv_parse(r)], 96)
        o = llm_generate(model, tok, [conv_A(memories[r.video_id], r)], CFG['llm']['max_new_tokens'])[0]
        llm_generate(model, tok, [conv_verify(memories[r.video_id], r, letter_of(parse_json(o)) or 'A')], 96)
    E2E['D_chain_s_per_q'] = round((time.time() - t0) / 10, 2)
    t0 = time.time()
    for r in rows[:10]:
        llm_generate(model, tok, [conv_A(memories[r.video_id], r)], CFG['llm']['max_new_tokens'])
    E2E['A_s_per_call_unbatched'] = round((time.time() - t0) / 10, 2)
    if tok is not None:
        E2E['M2_tokens_median'] = int(np.median([n_tokens(tok, m) for m in memories.values()]))
    E2E['reasoner'] = llm_spec['id']
    E2E['M2_words_per_video'] = E2E_MEM_WORDS
    del model, tok; free_memory()

def score(path):
    P_ = read_jsonl(path)
    return {'n': len(P_), 'accuracy': round(float(np.mean([p['pred'] == p['gold'] for p in P_])), 3) if P_ else None,
            'valid_json': round(float(np.mean([p['valid_json'] for p in P_])), 3) if P_ else None}
E2E['M2_A'] = score(PRED_A); E2E['blind'] = score(PRED_B)
save_section('e2e', E2E)
print(json.dumps(E2E, indent=1))
print('\nReminder: 50 pilot questions only show the pipeline works; these accuracies are not results.')

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

{
 "A_s_per_q_batched": 5.9,
 "blind_s_per_q_batched": 0.26,
 "D_chain_s_per_q": 9.82,
 "A_s_per_call_unbatched": 4.58,
 "M2_tokens_median": 842,
 "reasoner": "Qwen/Qwen3-8B",
 "M2_words_per_video": {
  "12392885583": 142,
  "3340340268": 240,
  "4094488636": 121,
  "4127497265": 146,
  "5406108447": 285,
  "5498331490": 321,
  "6762389913": 227,
  "7988210561": 95,
  "8505893258": 184,
  "9873067604": 183
 },
 "M2_A": {
  "n": 50,
  "accuracy": 0.44,
  "valid_json": 0.94
 },
 "blind": {
  "n": 50,
  "accuracy": 0.38,
  "valid_json": 1.0
 }
}

Reminder: 50 pilot questions only show the pipeline works; these accuracies are not results.


## 8. Projections and go/no-go rules

The measured costs are scaled up to the full plan, and the rules from Phase 0 are applied:

| If | Then |
| --- | --- |
| Qwen3-8B (4-bit) takes > 4 s per LLM call | main grid uses 400 of its 600 questions (a fixed, stratified subset); Tier 2 is cut first |
| Qwen3-8B does not load, or runs out of memory | Qwen3-4B becomes the main reasoner; the model comparison becomes Qwen3-4B vs Gemma 3 4B |
| Qwen3-VL-4B in fp16 gives NaN or inf | 4-bit Qwen3-VL-4B |
| Projected Tier 1 > 35 T4 hours | A-SC and D-X controls run on the dev set only; the descriptive set is dropped |

Decisions are written to `config.yaml` under `pilot_decisions`, and model revisions are pinned to exact commit hashes.

In [13]:
cfgv = CFG['cv']
s_frame = CV.get('s_per_frame_det_track', 0) + CV.get('s_per_frame_decode', 0)
s_window = EVENTS_STATS.get('s_per_window') or 0
def cv_hours(seconds_of_video):
    frames = seconds_of_video * cfgv['decode_fps']
    windows = max(0.0, (seconds_of_video - cfgv['window_s']) / cfgv['window_stride_s']) + 1
    return (frames * s_frame + windows * s_window) / 3600

nextqa_secs = sum(data_check['video_seconds'][n] for n in ['dev', 'main_grid', 'descriptive'])
star_manifest = P['data'] / 'star' / 'clip_manifest.csv'
if star_manifest.exists():
    Mf = pd.read_csv(star_manifest)
    star_secs = float((Mf.n_frames_src / Mf.fps + Mf.pause_s.fillna(0)).sum())
else:
    star_secs = 360 * 20 + 480 * 37                     # planning guess until 01b_star has run
s_call = E2E.get('A_s_per_call_unbatched') or 3.0
s_call_batched = E2E.get('A_s_per_q_batched') or s_call
d_per_q = E2E.get('D_chain_s_per_q') or 3.5 * s_call
grid_n = 600
def llm_hours(grid_n):
    calls_A_batched = 6 * grid_n + 360 * 2                          # E3 A cells + STAR A cells (batched)
    d_questions = 6 * grid_n + grid_n + 360 * 2 + 120 * 4           # E3 D cells, D-X, STAR D, pause/sham
    other = grid_n * (3 + 2 + 3) * s_call                           # B, C, A-SC (approx. as unbatched calls)
    # D re-answers after a failed verification; assume about 1 question in 6 does (factor 1.17)
    return (calls_A_batched * s_call_batched + d_questions * d_per_q * 1.17 + other) / 3600
proj = {'cv_hours': round(cv_hours(nextqa_secs) + cv_hours(star_secs), 1),
        'llm_tier1_hours_600': round(llm_hours(600), 1), 'llm_tier1_hours_400': round(llm_hours(400), 1)}
proj['tier1_total_hours'] = round(proj['cv_hours'] + proj['llm_tier1_hours_600'], 1)

decisions, notes = {}, []
if s_call > 4.0:
    decisions['main_grid_n'] = 400; grid_n = 400
    notes.append(f'Qwen3-8B at {s_call:.1f} s/call (> 4 s): main grid uses 400 questions')
if not MODEL_TESTS.get('reasoner_main', {}).get('status', '').startswith('ok'):
    decisions['reasoner_main'] = CFG['models']['reasoner_small']['id']
    notes.append('Qwen3-8B failed to load: Qwen3-4B is the main reasoner')
if MODEL_TESTS.get('vlm_events', {}).get('fp16_nan'):
    decisions['vlm_events_quant'] = 'nf4'
    notes.append('Qwen3-VL-4B overflows in fp16: 4-bit')
total = proj['cv_hours'] + (proj['llm_tier1_hours_400'] if grid_n == 400 else proj['llm_tier1_hours_600'])
if total > 35:
    decisions['controls_on_dev_only'] = True; decisions['drop_descriptive'] = True
    notes.append(f'projected Tier 1 {total:.1f} h (> 35 h): A-SC and D-X on dev only, descriptive set dropped')
if not notes:
    notes.append('all measurements within the plan: no changes')

# pin model revisions to exact commits
pins = {}
if not MOCK:
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        for k, spec in CFG['models'].items():
            try:
                pins[k] = api.model_info(spec['id'], revision=spec.get('revision', 'main')).sha
            except Exception as e:
                pins[k] = f'unresolved ({e.__class__.__name__})'
    except Exception as e:
        log(f'could not pin revisions: {e}', 'WARN')

# write decisions into config.yaml (existing values kept unless a rule changes them)
with open(CONFIG_PATH) as f:
    cfg_now = yaml.safe_load(f)
cfg_now['pilot_decisions'] = {'date': dt.date.today().isoformat(), **decisions, 'notes': notes}
if 'main_grid_n' in decisions:
    cfg_now['data']['main_grid_n'] = decisions['main_grid_n']
if 'reasoner_main' in decisions:
    cfg_now['models']['reasoner_main']['id'] = decisions['reasoner_main']
if 'vlm_events_quant' in decisions:
    cfg_now['models']['vlm_events']['quant'] = 'nf4'
for k, sha in pins.items():
    if isinstance(sha, str) and len(sha) == 40:
        cfg_now['models'][k]['revision'] = sha
cfg_now['pinned_versions'] = {k: v for k, v in versions.items() if v}
with open(CONFIG_PATH, 'w') as f:
    yaml.safe_dump(cfg_now, f, sort_keys=False)

save_section('projection', proj)
save_section('decisions', {'decisions': decisions, 'notes': notes, 'pins': pins})
RUN.finish('done', notes='; '.join(notes))
print(json.dumps(proj, indent=1))
print('\nDecisions:')
for n in notes:
    print(' -', n)
print(f'\nSaved: {FEAS_PATH}')

[08:40:44] INFO  run finished: 00b_pilot-20261016-081448-c6ae86 [done] 25.9 min, peak RAM 6.39 GB, peak VRAM 9.18 GB
{
 "cv_hours": 37.0,
 "llm_tier1_hours_600": 30.4,
 "llm_tier1_hours_400": 22.0,
 "tier1_total_hours": 67.4
}

Decisions:
 - Qwen3-8B at 4.6 s/call (> 4 s): main grid uses 400 questions
 - projected Tier 1 59.0 h (> 35 h): A-SC and D-X on dev only, descriptive set dropped

Saved: /content/drive/MyDrive/ST-Reason/outputs/pilot/feasibility.json


## What to send back

Download `outputs/pilot/feasibility.json` and send it, along with any cell that printed an error. That one file holds every measurement and decision, and it is what notebooks 02–05 are sized from.